In [1]:
import pandas as pd
from pathlib import Path

In [2]:
artifact_path = Path("../artifacts/01_order_level_table.parquet")

ml_table = pd.read_parquet(artifact_path)

print(f"Rows: {len(ml_table):,}")
print(f"Columns: {len(ml_table.columns)}")

Rows: 99,441
Columns: 14


In [3]:
ml_table[
    [
        "order_id",
        "order_status",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
].head(10)

,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-16 18:17:02,2018-02-26
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,2017-07-26 10:57:55,2017-08-01
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,NaT,2017-05-09
7,6514b8ad8028c9f2cc2374ded245783f,delivered,2017-05-26 12:55:51,2017-06-07
8,76c6e866289321a7c93b82b54852dc33,delivered,2017-02-02 14:08:10,2017-03-06
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,2017-08-16 17:14:30,2017-08-23


In [4]:
ml_table["late_delivery"] = (
    ml_table["order_delivered_customer_date"]
    > ml_table["order_estimated_delivery_date"]
).astype("int8")

In [5]:
ml_table[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "late_delivery",
    ]
].head(20)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,0
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,0
6,136cce7faa42fdb2cefd53fdc79a6098,NaT,2017-05-09,0
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,0
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,0


In [6]:
label_counts = ml_table["late_delivery"].value_counts()

print(label_counts)

late_delivery
0    91614
1     7827
Name: count, dtype: int64


In [7]:
label_distribution = (
    ml_table["late_delivery"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(label_distribution)

late_delivery
0    92.13
1     7.87
Name: proportion, dtype: float64


In [9]:
print(
    "Missing actual delivery date:",
    ml_table["order_delivered_customer_date"].isna().sum()
)

print(
    "Missing estimated delivery date:",
    ml_table["order_estimated_delivery_date"].isna().sum()
)

Missing actual delivery date: 2965
Missing estimated delivery date: 0


In [10]:
ml_table.loc[
    ml_table["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [11]:
# Keep only orders where an actual delivery date exists
labeled_table = ml_table[
    ml_table["order_delivered_customer_date"].notna()
].copy()

# Create target:
# 0 = delivered on time
# 1 = delivered late
labeled_table["late_delivery"] = (
    labeled_table["order_delivered_customer_date"]
    > labeled_table["order_estimated_delivery_date"]
).astype("int8")

print("Rows:", len(labeled_table))
print("Columns:", len(labeled_table.columns))

Rows: 96476
Columns: 15


In [12]:
labeled_table[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "late_delivery"
    ]
].head(10)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,0
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,0
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,0
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,0
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-29 11:18:31,2017-06-07,0


In [13]:
label_distribution = (
    labeled_table["late_delivery"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(label_distribution)

late_delivery
0    91.89
1     8.11
Name: proportion, dtype: float64


In [14]:
print(labeled_table["late_delivery"].value_counts())

late_delivery
0    88649
1     7827
Name: count, dtype: int64


In [15]:
labeled_table.to_parquet(
    "../artifacts/02_labeled_orders.parquet",
    index=False
)

print("Saved: ../artifacts/02_labeled_orders.parquet")

Saved: ../artifacts/02_labeled_orders.parquet
